In [0]:
_checkpoints = "dbfs:/Volumes/workspace/retails/raw_data/_checkpoints/retails/dev/gold/customers"

In [0]:
# -- # Read clean customers from silver
silver_df = (
    spark.readStream \
        .format("delta") \
        .option("readChangeFeed", "true") \
        .table("retails.silver.customers_cleaned")
)

In [0]:
from pyspark.sql.functions import col, current_timestamp, lit, sha2, concat_ws

# Prepare dataframe for gold SCD2
silver_df = (
    silver_df
        .select(
            "customer_id",
            "customer_fname",
            "customer_lname",
            "customer_email",
            "customer_street",
            "customer_city",
            "customer_state",
            "customer_zipcode",
            "op",
            "_change_type"
        ) \
            
        .filter(
            col("_change_type").isin(
                "insert",
                "update_postimage",
                "delete"
            )
        ) \

        .withColumn("is_active", lit(True).cast("boolean")) \
        .withColumn("effective_from", current_timestamp()) \
        .withColumn(
            "effective_to",
            lit(None).cast("timestamp")
        ) \
        .withColumn("is_current", lit(True).cast("boolean")) \
        .withColumn("created_ts", current_timestamp()) \
        .withColumn("updated_ts", current_timestamp()) \
        .withColumn("record_hash",
                        sha2(
                            concat_ws(
                                "||",
                                col("customer_id"),
                                col("customer_fname"),
                                col("customer_lname"),
                                col("customer_email"),
                                col("customer_street"),
                                col("customer_city"),
                                col("customer_state"),
                                col("customer_zipcode")
                            ),
                            256
                        )
                    ))
    


In [0]:
# STEP-1: Expire old current rows
MERGE_EXPIRE = """
MERGE INTO retails.gold.dim_customers t
USING silver_customers_vw s

ON t.customer_id = s.customer_id
AND t.is_current = true

WHEN MATCHED AND s.op = 'UPDATE'
THEN UPDATE SET
    t.is_active = false,
    t.is_current = false,
    t.effective_to = current_timestamp(),
    t.updated_ts = current_timestamp()

WHEN MATCHED AND s.op = 'DELETE'
THEN UPDATE SET 
    t.is_active = false,
    t.is_current = false,
    t.effective_to = current_timestamp(),
    t.updated_ts = current_timestamp()
"""

In [0]:
# STEP-2: Insert new versions
INSERT_NEW = """
INSERT INTO retails.gold.dim_customers(
    customer_id,
    customer_fname,
    customer_lname,
    customer_email,
    customer_street,
    customer_city,
    customer_state,
    customer_zipcode,
    is_active,
    effective_from,
    effective_to,
    is_current,
    created_ts,
    updated_ts)

SELECT
    s.customer_id,
    s.customer_fname,
    s.customer_lname,
    s.customer_email,
    s.customer_street,
    s.customer_city,
    s.customer_state,
    s.customer_zipcode,
    true,
    current_timestamp(),
    CAST(NULL AS TIMESTAMP),
    true,
    current_timestamp(),
    current_timestamp()

FROM silver_customers_vw s

WHERE s.op IN ('INSERT', 'UPDATE')
"""

In [0]:
# foreachBatch function
def upsert_to_gold(batch_df, batch_id):

    batch_df.createOrReplaceTempView(
        "silver_customers_vw"
    )

    spark.sql(MERGE_EXPIRE)

    spark.sql(INSERT_NEW)


In [0]:
# Start streaming query
query = (
    silver_df.writeStream
        .foreachBatch(upsert_to_gold)
        .option(
            "checkpointLocation",
            _checkpoints
        )
        .trigger(availableNow=True)
        .start()
)

query.awaitTermination()

In [0]:
# dbutils.fs.ls("dbfs:/Volumes/data/raw/_checkpoints/retails/dev/gold/customers/")
# dbutils.fs.rm("dbfs:/Volumes/data/raw/_checkpoints/retails/dev/gold/customers/", True)

In [0]:
%sql
-- make silver cleaned table CDF enabled

-- ALTER TABLE retails.silver.customers_cleaned
-- SET TBLPROPERTIES (
--     delta.enableChangeDataFeed = true
-- )

-- DESCRIBE TABLE EXTENDED retails.silver.customers_cleaned;

In [0]:
%sql
select * from retails.gold.dim_customers;